# Can the model compose rules it learned in context? — experiment 1 (composition), `openai/gpt-6-luna`

**Question.** If the model sees two rules separately, can it apply their composition, which it has never been
shown? If it really treats the rules as reusable parts, composing should be about as accurate as seeing the
composite directly.

**Two conditions on the same query grid** (neutral prompts: nothing about which operations exist):
* `compose`: 3 examples of **Rule P** (the first operation), then 3 examples of **Rule Q** (the second operation),
  then "Apply Rule P and then Rule Q to this grid". The composite is never shown.
* `direct` (holistic baseline): 3 examples of the composite transformation itself, then "Apply the same
  transformation to this grid" (as in notebooks 48-50).

**Pairs.** The 6 ordered pairs of distinct operations (AB, AC, BA, BC, CA, CB; A = shift right, B = rotate 90°
clockwise, C = mirror left-right; hidden from the model). The pairs do not commute, so order matters, and the
composite never equals P alone, Q alone or the reversed order. Every set of 3 examples determines its rule uniquely
among the 288 transformations.

**Size.** 10 tasks per pair × 6 pairs × 2 conditions = **120 calls** (~$0.1-0.2).

**Key measure.** Composition gap = accuracy(`compose`) − accuracy(`direct`). Wrong `compose` answers are classified
by what was applied instead: P only, Q only, the reversed order (Q then P), or something else.

Cell 4 = dry run (no cost). Cell 5 = 4-call pilot. Cell 6 = full run (resumable). Cell 7 = tables and zip.


In [1]:
# --- 2. Bootstrap: locate or clone the PRIVATE repository (credentials are never printed or stored) ---
import base64, importlib.util, os, pathlib, subprocess, sys

REPO_URL = "https://github.com/zhexuandliu/Decompositionality.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

def _find_repo():
    candidates = [pathlib.Path("/content/Decompositionality"), pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    for p in candidates:
        if (p / "src" / "decompositionality").is_dir():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        import getpass
        token = getpass.getpass("GitHub token with read access to the private repo (input hidden): ")
    # Pass the credential through environment-scoped git config (not the URL, not the command line,
    # not persisted in .git/config).
    _hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    _env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
                GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
    REPO_DIR = pathlib.Path("/content/Decompositionality" if IN_COLAB else "Decompositionality").resolve()
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                        env=_env, capture_output=True, text=True)
    del token, _hdr, _env
    if _r.returncode != 0:
        raise RuntimeError("git clone failed (check the GITHUB_TOKEN secret / repo access):\n" + _r.stderr[-2000:])

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for _mod in ("numpy", "pandas", "matplotlib", "torch"):
    if importlib.util.find_spec(_mod) is None:  # Colab already ships these; never reinstall torch/CUDA here
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _mod], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository:", REPO_DIR, "| commit:", REPO_COMMIT or "unknown")


Repository: /content/Decompositionality | commit: 457d041778308165ff24aec9d8b300ec2f27af65


In [2]:
# --- 3. Configuration ---------------------------------------------------------------------------------------------
MODEL = "openai/gpt-6-luna"
N_PER_PAIR = 20
N_DEMOS = 3                      # examples per rule (and for the direct baseline)
TASK_SEED = 0
CONDITIONS = ["compose", "direct"]
REASONING = None                 # None = default (medium for this model)
MAX_TOKENS = None
WORKERS = 128
USE_DRIVE = IN_COLAB
RESULTS_ROOT = os.environ.get("DECOMP_RESULTS_ROOT") or (
    "/content/drive/MyDrive/decompositionality_results" if USE_DRIVE else str(REPO_DIR / "results"))
# -----------------------------------------------------------------------------------------------------------------
if USE_DRIVE and IN_COLAB and RESULTS_ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
if IN_COLAB and not os.environ.get("OPENROUTER_API_KEY"):
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")      # never printed
print("OpenRouter key present:", bool(os.environ.get("OPENROUTER_API_KEY")))
import json, pathlib
import pandas as pd
from IPython.display import display
pd.set_option("display.max_rows", 300); pd.set_option("display.width", 220)
from decompositionality import icl_api as IA
_tag = "default" if not REASONING else "_".join(f"{k}{v}" for k, v in REASONING.items())
OUT_DIR = pathlib.Path(RESULTS_ROOT) / "icl_api" / f"compose_k2_{MODEL.replace('/', '_')}_demos{N_DEMOS}_n{N_PER_PAIR}_seed{TASK_SEED}_{_tag}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_JSONL = OUT_DIR / "responses.jsonl"
print("results ->", OUT_DIR)


Mounted at /content/drive
OpenRouter key present: True
results -> /content/drive/MyDrive/decompositionality_results/icl_api/compose_k2_openai_gpt-6-luna_demos3_n10_seed0_default


In [3]:
# --- 4. Dry run: build the tasks and show both prompts (no API calls) -----------------------------------------------
tasks = IA.make_compose_tasks(IA.K2_WORDS, n_per_pair=N_PER_PAIR, n_demos=N_DEMOS, seed=TASK_SEED)
(OUT_DIR / "tasks.json").write_text(json.dumps(tasks))
print(len(tasks), "tasks")
for c in CONDITIONS:
    print(f"\n==================== condition '{c}' (task {tasks[0]['task_id']}) ====================")
    print(IA.build_compose_prompt(tasks[0], c))
print("\ncalls planned:", len(tasks) * len(CONDITIONS))


60 tasks

==================== condition 'compose' (task AB-000) ====================
Below are examples of two hidden transformations, Rule P and Rule Q. Each example shows a 6x6 grid of digits (Input) and the grid obtained by applying that rule to it (Output).

Rule P

Example 1
Input:
0 8 2 4 7 7
5 3 8 6 4 5
0 5 0 4 5 1
2 6 9 2 1 0
4 3 2 6 6 1
0 9 4 1 0 9
Output:
7 0 8 2 4 7
5 5 3 8 6 4
1 0 5 0 4 5
0 2 6 9 2 1
1 4 3 2 6 6
9 0 9 4 1 0

Example 2
Input:
0 1 3 1 8 1
2 2 5 9 5 4
9 1 6 9 9 6
1 5 2 7 7 4
4 1 8 6 3 7
0 4 6 3 6 3
Output:
1 0 1 3 1 8
4 2 2 5 9 5
6 9 1 6 9 9
4 1 5 2 7 7
7 4 1 8 6 3
3 0 4 6 3 6

Example 3
Input:
7 1 2 2 9 4
9 9 8 9 1 9
4 7 5 9 1 7
0 9 1 0 5 0
7 3 3 6 9 9
9 4 8 9 4 1
Output:
4 7 1 2 2 9
9 9 9 8 9 1
7 4 7 5 9 1
0 0 9 1 0 5
9 7 3 3 6 9
1 9 4 8 9 4

Rule Q

Example 1
Input:
3 5 3 5 1 4
2 9 4 1 6 9
1 6 3 0 7 9
9 8 3 7 3 9
7 9 0 2 5 3
1 2 2 7 6 6
Output:
1 7 9 1 2 3
2 9 8 6 9 5
2 0 3 3 4 3
7 2 7 0 1 5
6 5 3 7 6 1
6 3 9 9 9 4

Example 2
Input:
5 2 1 4 0 5
3 2 7 2 9 3

In [4]:
# --- 5. Pilot: 4 calls --------------------------------------------------------------------------------------------------
IA.run(tasks[:2], OUT_JSONL, MODEL, experiments=CONDITIONS, hint=False, reasoning=REASONING, max_tokens=MAX_TOKENS,
       workers=WORKERS)
display(IA.score(tasks, OUT_JSONL))


0 calls already done; 4 to run (model openai/gpt-6-luna)
  4/4 calls finished (4 ok)


,task_id,word,k,experiment,parsed,correct,applied,applied_is_p_only,applied_is_q_only,applied_is_reversed,error_type,prompt_tokens,completion_tokens,reasoning_tokens,cost_usd,wrong_cells
0,AB-000,AB,2,compose,True,True,AB,False,False,False,correct,1082,703,619,0.000487,NaN
1,AB-001,AB,2,compose,True,True,AB,False,False,False,correct,1082,932,848,0.000601,NaN
2,AB-000,AB,2,direct,True,False,not a legal transform,NaN,NaN,NaN,execution slip,615,5747,5663,0.002935,2.0
3,AB-001,AB,2,direct,True,True,AB,False,False,False,correct,615,11221,11137,0.005672,NaN


In [5]:
# --- 6. Full run (resumable: calls already saved are skipped) --------------------------------------------------------
IA.run(tasks, OUT_JSONL, MODEL, experiments=CONDITIONS, hint=False, reasoning=REASONING, max_tokens=MAX_TOKENS,
       workers=WORKERS)


4 calls already done; 116 to run (model openai/gpt-6-luna)
  10/116 calls finished (10 ok)
  20/116 calls finished (20 ok)
  30/116 calls finished (30 ok)
  40/116 calls finished (40 ok)
  50/116 calls finished (50 ok)
  60/116 calls finished (60 ok)
  70/116 calls finished (70 ok)
  80/116 calls finished (80 ok)
  90/116 calls finished (90 ok)
  100/116 calls finished (100 ok)
  110/116 calls finished (110 ok)
  116/116 calls finished (116 ok)


116

In [6]:
# --- 7. Tables ---------------------------------------------------------------------------------------------------------
df = IA.score(tasks, OUT_JSONL)
acc = df.pivot_table(index="word", columns="experiment", values="correct", aggfunc="mean")
acc.loc["ALL"] = df.groupby("experiment").correct.mean()
acc["composition gap (compose - direct)"] = acc["compose"] - acc["direct"]
print("Accuracy by pair and condition:"); display(acc.round(3))
print("\nError types:"); display(df.groupby("experiment").error_type.value_counts(normalize=True).unstack(fill_value=0).round(3))
c = df[(df.experiment == "compose") & ~df.correct]
print(f"\nWrong 'compose' answers ({len(c)}): what was applied instead")
summary = {"P only": int(c.get("applied_is_p_only", pd.Series(dtype=bool)).fillna(False).sum()),
           "Q only": int(c.get("applied_is_q_only", pd.Series(dtype=bool)).fillna(False).sum()),
           "reversed order (Q then P)": int(c.get("applied_is_reversed", pd.Series(dtype=bool)).fillna(False).sum())}
summary["other"] = len(c) - sum(summary.values())
display(pd.Series(summary, name="count").to_frame())
display(c.groupby(["word", "applied"]).size().rename("count").reset_index().sort_values("count", ascending=False))
paired = df.pivot_table(index="task_id", columns="experiment", values="correct")
print("\nPaired outcome on the same query (rows: direct, columns: compose):")
display(pd.crosstab(paired["direct"], paired["compose"]))
print("\nUsage:"); display(df[["prompt_tokens", "completion_tokens", "reasoning_tokens", "cost_usd"]]
                           .apply(pd.to_numeric, errors="coerce").groupby(df.experiment).mean().round(1))
df.to_csv(OUT_DIR / "scores.csv", index=False)
print("saved to", OUT_DIR)
if IN_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
    files.download(str(OUT_DIR) + ".zip")


Accuracy by pair and condition:


experiment,compose,direct,composition gap (compose - direct)
word,,,
AB,0.9,0.7,0.2
AC,1.0,1.0,0.0
BA,0.8,0.9,-0.1
BC,1.0,1.0,0.0
CA,1.0,0.9,0.1
CB,0.7,0.9,-0.2
ALL,0.9,0.9,0.0



Error types:


error_type,correct,execution slip,other wrong grid,unparsed,wrong transform
experiment,,,,,
compose,0.9,0.033,0.05,0.000,0.017
direct,0.9,0.050,0.00,0.017,0.033



Wrong 'compose' answers (6): what was applied instead


/tmp/ipykernel_3112/912353391.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  summary = {"P only": int(c.get("applied_is_p_only", pd.Series(dtype=bool)).fillna(False).sum()),
/tmp/ipykernel_3112/912353391.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "Q only": int(c.get("applied_is_q_only", pd.Series(dtype=bool)).fillna(False).sum()),
/tmp/ipykernel_3112/912353391.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To

... (5 outputs left out)

,prompt_tokens,completion_tokens,reasoning_tokens,cost_usd
experiment,,,,
compose,1082.0,832.5,748.5,0.0
direct,615.0,2452.9,2369.4,0.0


saved to /content/drive/MyDrive/decompositionality_results/icl_api/compose_k2_openai_gpt-6-luna_demos3_n10_seed0_default


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>